In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
_r = pathlib.Path.cwd().resolve()
while not (_r / "config" / "settings.yaml").exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))                      # makes `import src...` work from any folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.notebook_utils import *
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 40, "display.width", 140)
print("project root:", _r)

project root: /app


# 06 - Model training & hyper-parameter tuning
Detailed look at randomised search for XGBoost, plus serialisation and metadata (what the pipeline does in its `train` and `persist` stages).

In [2]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from src.fraud_detection.train import candidate_models
fe = load_features(); d = split_xy(fe); X_tr, y_tr, _ = d['train']; X_va, y_va, _ = d['validation']
pw = (y_tr == 0).sum() / (y_tr == 1).sum(); est, space = candidate_models(CFG['seed'], pw, 2)['xgboost']

In [3]:
search = RandomizedSearchCV(est, space, n_iter=12, scoring='average_precision', cv=StratifiedKFold(3, shuffle=True, random_state=CFG['seed']), random_state=CFG['seed'], n_jobs=1, return_train_score=True).fit(X_tr, y_tr)
res = pd.DataFrame(search.cv_results_)[['mean_train_score','mean_test_score','std_test_score','param_max_depth','param_learning_rate','param_subsample']].sort_values('mean_test_score', ascending=False); res.round(4).head(8)

,mean_train_score,mean_test_score,std_test_score,param_max_depth,param_learning_rate,param_subsample
8,0.9314,0.7252,0.0134,6,0.0374,0.8440
1,0.8568,0.7251,0.0106,5,0.0305,0.7335
5,0.8882,0.7213,0.0134,6,0.0227,0.7530
3,0.8611,0.7165,0.0155,6,0.0200,0.8470
10,0.8777,0.7152,0.0110,4,0.0632,0.6739
6,0.9948,0.7116,0.0208,7,0.0708,0.8721
2,0.9999,0.7101,0.0180,7,0.1166,0.8888
11,0.9790,0.7063,0.0154,4,0.1632,0.8392


In [4]:
res.plot.scatter(x='mean_train_score', y='mean_test_score', title='Train vs CV score (overfitting check)'); plt.plot([0.3, 1], [0.3, 1], 'k--', lw=.8)

In [5]:
import joblib, json, tempfile, pathlib
best = search.best_estimator_; print(search.best_params_)
with tempfile.TemporaryDirectory() as t:
    p = pathlib.Path(t) / 'xgb.joblib'; joblib.dump(best, p); back = joblib.load(p)
    assert np.allclose(back.predict_proba(X_va)[:, 1], best.predict_proba(X_va)[:, 1]); print('serialisation round-trip OK,', p.stat().st_size // 1024, 'KiB')

{'colsample_bytree': np.float64(0.6063865008880857), 'learning_rate': np.float64(0.03737508643922314), 'max_depth': 6, 'min_child_weight': 7, 'subsample': np.float64(0.8439986631130484)}
serialisation round-trip OK, 608 KiB


In [6]:
print(json.dumps(json.loads((ROOT / 'models' / 'model_metadata' / 'xgboost.json').read_text()), indent=1)[:1500])

{
 "model": "xgboost",
 "artifact": "models/trained/xgboost.joblib",
 "best_params": {
  "colsample_bytree": 0.6624074561769746,
  "learning_rate": 0.03051362379613698,
  "max_depth": 5,
  "min_child_weight": 8,
  "subsample": 0.7334834444556088
 },
 "cv_score": 0.7251004709894305,
 "threshold": 0.9017223119735718,
 "train_seconds": 26.57737298400025,
 "features": [
  "log_amount",
  "hour",
  "day_of_week",
  "is_night",
  "is_weekend",
  "is_new_device",
  "is_foreign",
  "ip_risk_score",
  "velocity_1h",
  "log_account_age",
  "is_new_account",
  "cust_txn_count_prior",
  "amount_to_cust_avg_ratio",
  "log_secs_since_last",
  "new_device_x_foreign",
  "night_x_high_ratio",
  "high_risk_ip",
  "channel__mobile_app",
  "channel__web",
  "channel__pos",
  "channel__atm",
  "merchant_category__groceries",
  "merchant_category__restaurants",
  "merchant_category__travel",
  "merchant_category__electronics",
  "merchant_category__digital_goods",
  "merchant_category__utilities",
  "mercha

Search scoring uses **average precision** (area under PR curve) rather than accuracy, as it reflects performance on the rare positive class.